# Machine Learning & Deep Learning Notebook: Weeks 13–14
---
### **Curriculum Topics Covered:**
1. **Task 1:** Model Deployment Basics (`pickle` & `joblib` serialization)
2. **Task 2:** Neural Networks Basics (Perceptrons & Activation Functions)
3. **Task 3:** Mathematical Theory (Loss Functions & Optimizers)
4. **Task 4:** Implementing a 2-Layer Neural Network From Scratch using `NumPy`

## Task 1: Model Deployment Basics (`pickle` vs `joblib`)
Saving trained weights and states to disk for later inference or deployment pipelines.

In [ ]:
import pickle
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression

# Create dummy data
X_dummy = np.array([[1, 2], [2, 3], [3, 4], [4, 5]])
y_dummy = np.array([0, 0, 1, 1])

# Train simple model
model = LogisticRegression()
model.fit(X_dummy, y_dummy)

# Method 1: Using joblib (Optimized for large arrays)
joblib.dump(model, 'logistic_model.joblib')
loaded_joblib = joblib.load('logistic_model.joblib')
print("Joblib Model loaded successfully. Classes:", loaded_joblib.classes_)

# Method 2: Using standard pickle
with open('logistic_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('logistic_model.pkl', 'rb') as f:
    loaded_pickle = pickle.load(f)
print("Pickle Model loaded successfully. Intercept:", loaded_pickle.intercept_)

: 

## Tasks 2, 3 & 4: Deep Learning Foundations & NumPy Implementation From Scratch
Building a 2-layer Neural Network capable of learning non-linear structures (like the XOR logic problem) completely from scratch.

In [ ]:
class NNFromScratch:
    def __init__(self, input_dim, hidden_dim, output_dim, lr=0.1):
        self.lr = lr
        
        # Initialize parameters with small random values
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.1
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.1
        self.b2 = np.zeros((1, output_dim))
        
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))
        
    def sigmoid_derivative(self, a):
        return a * (1 - a)
        
    def forward(self, X):
        # Input layer -> Hidden layer
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = self.sigmoid(self.Z1)
        
        # Hidden layer -> Output layer
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.A2 = self.sigmoid(self.Z2)
        return self.A2
        
    def backward(self, X, y, output):
        m = X.shape[0]
        
        # Vectorized backpropagation
        dZ2 = output - y
        dW2 = (1 / m) * np.dot(self.A1.T, dZ2)
        db2 = (1 / m) * np.sum(dZ2, axis=0, keepdims=True)
        
        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * self.sigmoid_derivative(self.A1)
        dW1 = (1 / m) * np.dot(X.T, dZ1)
        db1 = (1 / m) * np.sum(dZ1, axis=0, keepdims=True)
        
        # SGD Parameter Updates
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        
    def train(self, X, y, epochs=2000):
        print("--- Starting Optimization Pipeline ---")
        for epoch in range(epochs):
            out = self.forward(X)
            self.backward(X, y, out)
            
            if epoch % 400 == 0:
                loss = np.mean((y - out) ** 2) # MSE Loss
                print(f"Epoch {epoch:4d} | Mean Squared Error Loss: {loss:.6f}")

### Training the network on an XOR dataset configuration

In [ ]:
# Setup XOR data matrices
X_xor = np.array([[0,0], [0,1], [1,0], [1,1]])
y_xor = np.array([[0], [1], [1], [0]])

# Create, train and evaluate network
nn = NNFromScratch(input_dim=2, hidden_dim=4, output_dim=1, lr=0.5)
nn.train(X_xor, y_xor, epochs=2001)

print("\n--- Verification Inference Predictions ---")
final_predictions = nn.forward(X_xor)
for inp, pred, truth in zip(X_xor, final_predictions, y_xor):
    print(f"Input Feature Vect: {inp} | Model Sigmoid Conf: {pred[0]:.4f} | Final Label: {1 if pred[0]>0.5 else 0} | Ground Truth: {truth[0]}")